# Module 7：跨系统维表 — 矿井 / 客户 / 物料

## 痛点故事

> 各系统的矿井、客户、物料编码字面上已经统一（如矿井都用 `M001`），
> 但字段名不统一——PI 用 `mine`，LIMS 用 `MINE_CODE`，SAP 用 `WERKS`。

跨系统产销分析时，JOIN 条件变成一堆 `WHERE` 映射：

```sql
-- 无维表时的痛苦 JOIN
SELECT s.VBELN, s.KUNNR, p.mine, l.MINE_CODE
FROM dwd_vbak s
JOIN dwd_pi_tags p ON s.WERKS = p.face   -- 字段名不同，要映射
JOIN dwd_lims_samples l ON p.mine = l.MINE_CODE  -- 再次映射
WHERE s.KUNNR = "K001";  -- 客户也逃不掉
```

维表统一字段名后，同样的查询变成：

```sql
-- 有维表后的清晰 JOIN
SELECT s.VBELN, c.customer_name, m.mine_name
FROM dwd_vbak s
JOIN dim_customer c ON s.KUNNR = c.kunnr
JOIN dim_mine m ON s.WERKS = m.sap_mine_field;
```

**本模块目标**：从 DWD 层聚合数据，构建 3 张维表（dim_mine / dim_customer / dim_material）。

In [ ]:
# -- Setup
import subprocess
import os

ROOT = "/home/szs/Playground/dg-demo"
os.chdir(ROOT)

print("Module 7: Dimension Tables")
print("=" * 60)

In [ ]:
# -- 1. 构建矿井维表 dim_mine
# 从 PI (dwd_tags.mine) 和 LIMS (dwd_samples.MINE_CODE) 提取矿井数据

result = subprocess.run(
    ["uv", "run", "python", "scripts/build_dimension_tables.py", "--dimension", "mine"],
    capture_output=True, text=True, cwd=ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

# 验证输出
from deltalake import DeltaTable
dt = DeltaTable.from_path(ROOT + "/data/lakehouse/dwd/_dimensions/dim_mine")
df = dt.to_pyarrow_table().to_pandas()
print(f"dim_mine 行数: {len(df)}")
print(df.to_string())

In [ ]:
# -- 2. 构建客户维表 dim_customer
# 从 SAP KNA1 提取客户数据

result = subprocess.run(
    ["uv", "run", "python", "scripts/build_dimension_tables.py", "--dimension", "customer"],
    capture_output=True, text=True, cwd=ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

dt = DeltaTable.from_path(ROOT + "/data/lakehouse/dwd/_dimensions/dim_customer")
df = dt.to_pyarrow_table().to_pandas()
print(f"dim_customer 行数: {len(df)}")
print(df.head(5).to_string())

In [ ]:
# -- 3. 构建物料维表 dim_material
# 从 SAP VBAP (MATNR) 提取物料数据
# 注：MARA 物料主数据暂不可用，mat_type 从 matnr 前4位推断

result = subprocess.run(
    ["uv", "run", "python", "scripts/build_dimension_tables.py", "--dimension", "material"],
    capture_output=True, text=True, cwd=ROOT
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

dt = DeltaTable.from_path(ROOT + "/data/lakehouse/dwd/_dimensions/dim_material")
df = dt.to_pyarrow_table().to_pandas()
print(f"dim_material 行数: {len(df)}")
print(df.head(5).to_string())

## JOIN 对比：有维表 vs 无维表

以下 SQL 对比展示了维表如何简化跨系统 JOIN。

In [ ]:
# -- 无维表：复杂 WHERE 映射
# PI 的 mine 字段、LIMS 的 MINE_CODE、SAP VBAK 的 WERKS 都是矿井编码
# 但字段名不同，JOIN 时必须逐个映射

sql_without_dim = """
-- 痛点：每次 JOIN 矿井都要映射字段名
SELECT
    v.VBELN,
    k.NAME1        AS customer_name,
    p.mine         AS mine_code,
    l.MINE_NAME    AS mine_name
FROM dwd_vbak v
-- 客户：SAP KUNNR 直接可 JOIN
JOIN dwd_kna1 k ON v.KUNNR = k.KUNNR
-- 矿井：PI mine -> LIMS MINE_CODE -> 才能 JOIN VBAK WERKS
JOIN dwd_tags p ON v.WERKS = p.face
JOIN dwd_samples l ON p.mine = l.MINE_CODE
WHERE v.KUNNR = "K00001";
"""
print("=== 无维表 SQL（痛苦版）===")
print(sql_without_dim)

In [ ]:
# -- 有维表：简洁 JOIN

sql_with_dim = """
-- 维表统一了各系统的矿井/客户/物料编码
SELECT
    v.VBELN,
    c.customer_name,
    m.mine_name
FROM dwd_vbak v
-- 客户维表：kunnr -> customer_name, region
JOIN dim_customer c ON v.KUNNR = c.kunnr
-- 矿井维表：sap_mine_field = WERKS，pi_mine_field = mine，lims_mine_field = MINE_CODE
JOIN dim_mine m ON v.WERKS = m.sap_mine_field;
"""
print("=== 有维表 SQL（清晰版）===")
print(sql_with_dim)

## 维表汇总

| 维表 | 源系统 | 路径 | 近似行数 |
|------|--------|------|------|
| dim_mine | PI + LIMS | `data/lakehouse/dwd/_dimensions/dim_mine/` | 5 |
| dim_customer | SAP KNA1 | `data/lakehouse/dwd/_dimensions/dim_customer/` | 15000 |
| dim_material | SAP VBAP | `data/lakehouse/dwd/_dimensions/dim_material/` | 7 |

**设计决策**：
- 维表数据来自 DWD 层（已清洗、统一编码）
- 存储为 Delta Lake，支持时间旅行
- 独立函数构建，互不依赖，可按需重建单张表

## 免责声明

> **诚实声明**：本 notebook 仅用于教学演示目的。
>
> - 维表数据来源于 Demo 环境模拟数据，不代表生产系统真实数据
> - 物料维表使用 VBAP MATNR 替代缺失的 MARA 主数据，mat_desc 和 mat_type 仅为演示推断
> - 本模块不构成生产数据治理流程的完整实现
>
> **扩展方向**：接入真实 SAP MARA 物料主数据、完善 credit_level 映射规则、实现增量更新